In [1]:
from keys import openai_key, replicate_key, groq_key
from working import JudgedDebate, get_client, Debater, Judge, CollabDebate

In [2]:
from sam_dictionaries.sae import AutoEncoder
import torch

def load_sae(
    layer: int
):
    path = f"./sam_dictionaries/resid_out_layer{layer}/10_32768/ae.pt"

    activation_dim = 512 # dimension of the NN's activations to be autoencoded
    dictionary_size = 64 * activation_dim # number of features in the dictionary
    ae = AutoEncoder(activation_dim, dictionary_size)
    ae.load_state_dict(torch.load(path))
    ae.to("cuda:0")

    return ae

In [3]:
from working import ActivationCache
from nnsight import LanguageModel
from working.config import CacheConfig

layer = 5

# TODO
# 5_554
# 4_4365
# 3_11662
# 2_7440
# 1_7703
# 0_8313

model = LanguageModel("EleutherAI/pythia-70m", device_map="auto", dispatch=True)
sae = load_sae(
    layer=layer
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [4]:
cfg = CacheConfig

cache = ActivationCache(
    layer=layer,
    model=model,
    sae=sae,
    cfg = cfg
)

100%|██████████| 9/9 [00:30<00:00,  3.44s/it]

Activation Cache Size: torch.Size([1800, 128, 32768])


In [5]:
feature_id = 554 
examples_list, max_act = cache.get_top_examples(feature_id)

n_debaters = 2

debaters = [
    Debater(
        get_client("openai", openai_key),
        f"debater {id}"
    ) 
    for id in range(n_debaters)
]

judge = Judge(
    get_client("openai", openai_key)
)

debate = JudgedDebate(debaters, judge, examples_list)

TypeError: PromptBuilder.__init__() missing 1 required positional argument: 'top_logits'

In [ ]:
debate.run(max_rounds=1)

debate.history.save("openai", f"judged_{layer}_{feature_id}")

['\n\nFirst, examine how the pronouns are used in the text examples.\n[QUOTE]: <unverified>They blacklisted competitors</unverified>\n[QUOTE]: <unverified>I guessed that someone was burning refuse.</unverified>\nThe pronouns “They” and “I” appear right before a descriptive action or detail, suggesting the start of a narrative passage.\n\nNext, consider the use of temporal adverbs in the examples.\n[QUOTE]: <unverified>I already had mixtapes</unverified>\n[QUOTE]: <unverified>When I first started road-testing online dating</unverified>\nThe temporal adverbs “already” and “first” point out specific timing or sequence in the narrative.\n\nAdditionally, explore how these pronouns and adverbs maintain a context of storytelling.\n[QUOTE]: <unverified>the minimum wage, they understood its impact</unverified>\n[QUOTE]: <verified>ordered a few dishes</verified>\nPronouns like “they” and phrases describing actions integrate into a narrative format, suggesting continuity in describing actions tak